# nb_01 — Bronze: ingest reference masters & the pay-grid feed

**Module 1 (notebook half).** The pipeline handles the flat monthly workforce-event
extracts. This notebook handles the sources a Copy activity handles badly: the
**nested JSON pay-grid feed** (arrays of band history) plus FX and the reference
masters.

**Prereqs**
- Schema-enabled lakehouse `lh_meridian_hr` attached as the default lakehouse.
- `BASE` below points at your fork's raw GitHub `/data` folder.

Everything lands in the `bronze` schema — a faithful copy of the source.

> **Lab notebook.** This is the fill-in-the-blank companion to the solution notebook of the same name. Each code cell only contains `# TODO` comments — use the markdown cell above each one to figure out what to build.


## Configuration and Spark setup

**Summary.** Sets the raw source base URL, the landing folder, and the lakehouse name; enables V-Order for Direct Lake; and ensures the `bronze` schema exists.


In [ ]:
# TODO: Configure Spark and confirm the bronze schema.
# - Set BASE to your fork's raw GitHub /data folder, LANDING to the lakehouse-relative
#   landing folder, and LH to the lakehouse name.
# - Import the PySpark functions module (commonly aliased as F).
# - Turn on V-Order (spark.conf.set) so Delta files are optimized for Direct Lake reads.
# - Create the `bronze` schema if it doesn't exist yet, and print BASE for confirmation.


## 1. Download the feeds into `Files/landing`

Download the hosted files from the raw GitHub repository into the lakehouse file
area. In production, the pipeline lands these files.


## Download the feeds into `Files/landing`

**Summary.** Downloads each hosted feed into the lakehouse file area, guarding against HTML error pages and validating JSON before writing.


In [ ]:
# TODO: Download each hosted feed into the lakehouse's Files/landing folder.
# - Compute the local mount path (/lakehouse/default/Files/landing) and create it.
# - Write a small `fetch(relative_path)` helper that builds the source URL from BASE,
#   downloads the bytes, and raises a clear error if the request fails.
# - Guard against accidentally downloading an HTML error page instead of real data, and
#   validate JSON payloads before writing them to disk.
# - Write the bytes to the landing folder and print the destination + size.
# - Loop over the pay-grid feed, FX rates, and the three reference masters.


## 2. Pay-grid feed — explode the nested band history

One object per (classification group, level) with a `band_history` array. We read
it as multiline JSON and `explode` the array into one row per (group, level,
effective_date). This Bronze table is the **source for the SCD Type 2 pay-band
dimension** in Module 3.


## Pay-grid feed — explode the nested band history

**Summary.** Reads the multiline JSON pay-grid feed and explodes its nested arrays into one flat row per (group, level, effective date), writing `bronze.pay_bands` — the source for the SCD2 pay-band dimension.


In [ ]:
# TODO: Explode the nested pay-grid JSON into a flat Bronze table.
# - Read the pay-grid feed as multiline JSON and check the expected top-level column
#   exists (fail with a helpful message if not — usually means BASE is wrong).
# - Explode the classifications array, then explode each classification's band_history
#   array so you get one row per (group, level, effective_date).
# - Cast the effective date and band_min/mid/max, and stamp an _ingested_at timestamp.
# - Overwrite bronze.pay_bands with the result and print a row count plus a small preview
#   for one classification's history.


## 3. FX and reference masters (flat CSV → Bronze Delta)


## FX and reference masters (flat CSV to Bronze Delta)

**Summary.** Defines a small helper that lands a header CSV into a Delta table with an ingest timestamp, then applies it to FX rates and the three reference masters.


In [ ]:
# TODO: Land the FX and reference-master CSVs into Bronze Delta tables.
# - Write a small `land_csv(path, table)` helper that reads a header CSV with schema
#   inference, adds an _ingested_at timestamp, and overwrites the target Delta table.
# - Print the table name and row count after each write.
# - Call the helper for fx_rates, workers, workers_delta, and cost_centers.


## 4. Confirm the pipeline landed the events

The pipeline runs `nb_00_setup_lakehouse` to create the target table and
`nb_00_setup_watermark` to initialize its last-loaded month. It then appends only the
months after that watermark to `bronze.workforce_events_raw` and advances the
watermark after every Copy succeeds. The fallback below downloads the files from
GitHub only if the target table does not exist.


## Confirm the pipeline landed the events (with fallback)

**Summary.** Checks that the pipeline-loaded `bronze.workforce_events_raw` exists; if it does not, downloads the monthly event CSVs from GitHub and loads them so the rest of the lab can proceed.


In [ ]:
# TODO: Confirm the pipeline-loaded events exist, with a fallback loader.
# - Try reading bronze.workforce_events_raw and print its row count.
# - If the table doesn't exist (catch the analysis exception), download each monthly
#   workforce_events_YYYY-MM.csv from GitHub for 2021-01 through 2025-12.
# - Read the downloaded CSVs together, add _source_file and _ingested_at columns, and
#   overwrite bronze.workforce_events_raw.
# - Print how many rows the fallback loaded.


## List the Bronze tables

**Summary.** Prints the tables now present in the `bronze` schema as a quick completion check.


In [ ]:
# TODO: List the tables now present in the bronze schema as a quick completion check.
